# Nebula Learning Galaxy — Bayesian Mastery Analysis

Demonstrates the Beta-Binomial mastery model that powers the app's adaptive learning.

**Pipeline**:  
`Quiz interaction → Beta update → Mastery P(θ) → Node colours + Recommendations`

**Model**: Bayesian Beta-Binomial with slip/guess factors  
- `α` counts evidence of mastery  
- `β` counts evidence of non-mastery  
- `P(mastery) = α / (α + β)`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
from scipy import stats
from scipy.special import betaln

np.random.seed(42)
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.facecolor': '#0a0e24',
    'figure.facecolor': '#060818',
    'axes.edgecolor': '#2a3050',
    'grid.color': '#1a2040',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlesize': 13,
    'font.family': 'monospace',
})

ACCENT  = '#60a5fa'
GREEN   = '#4ade80'
RED     = '#f87171'
PURPLE  = '#a78bfa'
ORANGE  = '#fb923c'

print('✅ Libraries loaded')

## 1. Model Definition

The model mirrors `backend/services/bayesian_service.py` exactly.

In [ ]:
# ── Model hyperparameters (mirror of bayesian_service.py) ────────────────
P_GUESS        = 0.25   # prob of guessing correctly without knowledge
P_SLIP         = 0.20   # prob of making a mistake despite knowledge
NEBULA_PRIOR   = 0.72   # UTD students start with 72% prior mastery
PRIOR_TOTAL    = 10     # prior pseudo-count strength

ALPHA_PRIOR = max(0.5, NEBULA_PRIOR * PRIOR_TOTAL)
BETA_PRIOR  = max(0.5, PRIOR_TOTAL - ALPHA_PRIOR)

print(f'Prior: α={ALPHA_PRIOR:.1f}, β={BETA_PRIOR:.1f}, P(mastery)={ALPHA_PRIOR/(ALPHA_PRIOR+BETA_PRIOR):.2%}')


def update_mastery(alpha: float, beta: float, success: bool, difficulty: float) -> tuple:
    """Bayesian Beta-Binomial update with slip/guess correction."""
    w_success = 1.0 - P_GUESS * (1.0 - difficulty)   # harder → larger update
    w_failure = 1.0 - P_SLIP  * difficulty             # harder → smaller penalty
    if success:
        alpha_new = alpha + w_success
        beta_new  = beta
    else:
        alpha_new = alpha
        beta_new  = beta + w_failure
    mastery = alpha_new / (alpha_new + beta_new)
    return alpha_new, beta_new, mastery


def mastery_probability(alpha: float, beta: float) -> float:
    return alpha / (alpha + beta)


# Test a quick update
a, b = ALPHA_PRIOR, BETA_PRIOR
a2, b2, m = update_mastery(a, b, success=True, difficulty=0.6)
print(f'After 1 correct answer (diff=0.6): α={a2:.2f}, β={b2:.2f}, P={m:.3f}')
a3, b3, m3 = update_mastery(a2, b2, success=False, difficulty=0.6)
print(f'After 1 wrong answer  (diff=0.6): α={a3:.2f}, β={b3:.2f}, P={m3:.3f}')

## 2. Synthetic Dataset Generation

Simulate **2 000 students × 15 topics** with realistic learning trajectories.

In [ ]:
N_STUDENTS  = 2000
N_TOPICS    = 15
MAX_ATTEMPTS = 12   # quiz attempts per student per topic

TOPICS = [
    ('CS 3345 — Trees',           0.55, 0.12),   # (name, avg_true_mastery, std)
    ('CS 3345 — Graphs',          0.50, 0.14),
    ('CS 3345 — Heaps',           0.60, 0.11),
    ('CS 3345 — Hashing',         0.65, 0.10),
    ('CS 3345 — Sorting',         0.70, 0.09),
    ('CS 3345 — DP Intro',        0.45, 0.15),
    ('CS 3341 — Probability',     0.50, 0.13),
    ('CS 3341 — Distributions',   0.48, 0.14),
    ('CS 3341 — Hypothesis',      0.42, 0.16),
    ('CS 4349 — NP Hardness',     0.40, 0.17),
    ('CS 4349 — Greedy',          0.58, 0.12),
    ('CS 4349 — Divide&Conquer',  0.62, 0.11),
    ('PHYS 2326 — Waves',         0.55, 0.13),
    ('MATH 2418 — Eigenvalues',   0.47, 0.16),
    ('MATH 2418 — SVD',           0.43, 0.18),
]

records = []

for student_id in range(N_STUDENTS):
    # Each student has a general aptitude factor
    aptitude = np.random.beta(4, 2)   # right-skewed: most students are decent

    for topic_idx, (topic_name, base_mastery, mastery_std) in enumerate(TOPICS):
        # Student's latent true mastery for this topic
        true_mastery = float(np.clip(
            np.random.normal(base_mastery * aptitude + 0.1 * (1 - aptitude), mastery_std),
            0.05, 0.98
        ))

        # Start from prior
        alpha, beta = ALPHA_PRIOR, BETA_PRIOR
        n_attempts  = np.random.randint(2, MAX_ATTEMPTS + 1)

        # Simulate quiz attempts with increasing difficulty over time
        for attempt in range(n_attempts):
            # Difficulty ramps up as mastery grows (adaptive)
            current_mastery = mastery_probability(alpha, beta)
            difficulty = float(np.clip(
                current_mastery * 0.7 + np.random.normal(0, 0.12),
                0.1, 0.95
            ))

            # P(correct) = true_mastery × (1 - P_SLIP) + (1 - true_mastery) × P_GUESS
            p_correct = true_mastery * (1 - P_SLIP) + (1 - true_mastery) * P_GUESS
            # Harder questions are tougher even for masters
            p_correct *= (1.0 - 0.15 * difficulty)
            success = bool(np.random.random() < p_correct)

            alpha, beta, mastery_est = update_mastery(alpha, beta, success, difficulty)

            records.append({
                'student_id':      student_id,
                'topic':           topic_name,
                'topic_idx':       topic_idx,
                'attempt':         attempt,
                'aptitude':        round(aptitude, 3),
                'true_mastery':    round(true_mastery, 4),
                'difficulty':      round(difficulty, 3),
                'success':         int(success),
                'alpha':           round(alpha, 4),
                'beta':            round(beta, 4),
                'mastery_est':     round(mastery_est, 4),
            })

df = pd.DataFrame(records)
print(f'Generated {len(df):,} quiz events  |  {df["student_id"].nunique()} students  |  {df["topic"].nunique()} topics')
df.head(8)

## 3. Beta Distribution Visualisation

How the Beta distribution evolves with evidence.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
theta = np.linspace(0, 1, 300)

scenarios = [
    ('Prior (no evidence)',       ALPHA_PRIOR, BETA_PRIOR, PURPLE),
    ('After 3 correct answers',   ALPHA_PRIOR+2.25, BETA_PRIOR, GREEN),
    ('After 5 correct + 2 wrong', ALPHA_PRIOR+3.75, BETA_PRIOR+1.6, ACCENT),
]

for ax, (title, a, b, col) in zip(axes, scenarios):
    y = stats.beta.pdf(theta, a, b)
    ax.fill_between(theta, y, alpha=0.25, color=col)
    ax.plot(theta, y, color=col, lw=2)
    mode = (a - 1) / (a + b - 2) if a > 1 and b > 1 else a / (a + b)
    mean = a / (a + b)
    ax.axvline(mean, color=col, ls='--', lw=1.2, alpha=0.8, label=f'Mean={mean:.2f}')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('θ (mastery probability)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1)

plt.suptitle('Beta-Binomial Posterior: Prior → Evidence Updates', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Learning Trajectories

How estimated mastery evolves over quiz attempts for sample students.

In [ ]:
topic = 'CS 3345 — Trees'
sample_students = df[df['topic'] == topic]['student_id'].drop_duplicates().sample(8, random_state=1).tolist()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: individual trajectories
cmap = cm.get_cmap('plasma', len(sample_students))
for i, sid in enumerate(sample_students):
    sdf = df[(df['student_id'] == sid) & (df['topic'] == topic)].sort_values('attempt')
    true_m = sdf['true_mastery'].iloc[0]
    ax1.plot(sdf['attempt'], sdf['mastery_est'], color=cmap(i), lw=1.8,
             alpha=0.85, label=f'S{sid} (true={true_m:.2f})')
    ax1.axhline(true_m, color=cmap(i), ls=':', lw=0.8, alpha=0.4)

ax1.set_title(f'{topic}\nIndividual Learning Curves')
ax1.set_xlabel('Quiz Attempt')
ax1.set_ylabel('Estimated Mastery P(θ)')
ax1.axhline(0.8, color=GREEN,  ls='--', lw=1, label='Expert threshold (0.80)')
ax1.axhline(0.5, color=ORANGE, ls='--', lw=1, label='Proficient threshold (0.50)')
ax1.legend(fontsize=7, loc='lower right')
ax1.set_ylim(0.3, 1.0)

# Right: average trajectory across all students, split by aptitude quartile
last_df = df[df['topic'] == topic].groupby('student_id').agg(
    final_mastery=('mastery_est', 'last'),
    true_mastery=('true_mastery', 'first'),
    n_attempts=('attempt', 'max'),
    aptitude=('aptitude', 'first'),
).reset_index()

quartiles = pd.qcut(last_df['aptitude'], 4, labels=['Q1 Low','Q2','Q3','Q4 High'])
last_df['aptitude_q'] = quartiles
colors_q = [RED, ORANGE, ACCENT, GREEN]
for q, col in zip(['Q1 Low','Q2','Q3','Q4 High'], colors_q):
    grp = last_df[last_df['aptitude_q'] == q]
    ax2.scatter(grp['true_mastery'], grp['final_mastery'],
                alpha=0.35, s=18, color=col, label=q)

ax2.plot([0, 1], [0, 1], 'w--', lw=1, alpha=0.4, label='Perfect calibration')
ax2.set_title('True Mastery vs Estimated Mastery\n(by Aptitude Quartile)')
ax2.set_xlabel('True Mastery (latent)')
ax2.set_ylabel('Model Estimate P(θ)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 5. Effect of Question Difficulty

Harder questions carry more information (larger gradient signal).

In [ ]:
diff_bins = pd.cut(df['difficulty'], bins=[0, 0.33, 0.66, 1.0], labels=['Easy', 'Medium', 'Hard'])
df['difficulty_bin'] = diff_bins

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
colors_d = [GREEN, ACCENT, RED]
labels_d = ['Easy (<0.33)', 'Medium (0.33–0.66)', 'Hard (>0.66)']

for ax, (bin_label, col, lbl) in zip(axes, zip(['Easy','Medium','Hard'], colors_d, labels_d)):
    grp = df[df['difficulty_bin'] == bin_label]

    # Weight of update when successful
    diffs = grp['difficulty'].values
    w_success = 1.0 - P_GUESS * (1.0 - diffs)
    w_failure = 1.0 - P_SLIP * diffs

    # Mastery change per event
    correct_grp  = grp[grp['success'] == 1]
    wrong_grp    = grp[grp['success'] == 0]

    ax.hist(1.0 - P_GUESS * (1.0 - correct_grp['difficulty']),
            bins=25, color=col, alpha=0.7, label='Correct weight')
    ax.hist(1.0 - P_SLIP * wrong_grp['difficulty'],
            bins=25, color=ORANGE, alpha=0.5, label='Wrong weight')
    ax.set_title(f'{lbl}')
    ax.set_xlabel('Update Weight')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('Slip/Guess Correction: Update Weight by Difficulty', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('\nMean update weight per difficulty bin:')
df['update_w'] = df.apply(
    lambda r: (1 - P_GUESS*(1-r['difficulty'])) if r['success'] else (1 - P_SLIP*r['difficulty']), axis=1
)
print(df.groupby('difficulty_bin')['update_w'].agg(['mean','std']).round(3))

## 6. Topic-Level Mastery Distribution

Final mastery distribution across all students for each topic.

In [ ]:
final_per_student = df.groupby(['student_id', 'topic']).agg(
    final_mastery=('mastery_est', 'last'),
    true_mastery=('true_mastery', 'first'),
    n_attempts=('attempt', 'max'),
).reset_index()

topic_stats = final_per_student.groupby('topic').agg(
    mean_mastery=('final_mastery', 'mean'),
    pct_expert=  ('final_mastery', lambda x: (x >= 0.8).mean()),
    pct_proficient=('final_mastery', lambda x: (x >= 0.5).mean()),
    mean_attempts=('n_attempts', 'mean'),
).round(3).sort_values('mean_mastery', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Horizontal bar chart of mean mastery
col_scale = cm.RdYlGn(np.linspace(0.3, 0.9, len(topic_stats)))
bars = ax1.barh(range(len(topic_stats)), topic_stats['mean_mastery'], color=col_scale, height=0.7)
ax1.set_yticks(range(len(topic_stats)))
ax1.set_yticklabels([t.split('—')[-1].strip() for t in topic_stats.index], fontsize=9)
ax1.axvline(0.8, color=GREEN,  ls='--', lw=1.2, label='Expert')
ax1.axvline(0.5, color=ORANGE, ls='--', lw=1.2, label='Proficient')
ax1.set_xlabel('Mean Estimated Mastery')
ax1.set_title('Average Mastery by Topic')
ax1.legend(fontsize=9)
ax1.set_xlim(0.3, 1.0)

# Violin plot for top 5 and bottom 5 topics
top5    = topic_stats.head(5).index.tolist()
bottom5 = topic_stats.tail(5).index.tolist()
selected = top5 + bottom5

data_for_violin = [
    final_per_student[final_per_student['topic'] == t]['final_mastery'].values
    for t in selected
]
parts = ax2.violinplot(data_for_violin, positions=range(len(selected)), showmedians=True)
for i, (body, t) in enumerate(zip(parts['bodies'], selected)):
    body.set_alpha(0.65)
    body.set_facecolor(GREEN if t in top5 else RED)
ax2.set_xticks(range(len(selected)))
ax2.set_xticklabels([t.split('—')[-1].strip() for t in selected], rotation=40, ha='right', fontsize=8)
ax2.set_ylabel('Final Mastery Distribution')
ax2.set_title('Top 5 (green) vs Bottom 5 (red) Topics')
ax2.axhline(0.8, color=GREEN, ls='--', lw=1)
ax2.axhline(0.5, color=ORANGE, ls='--', lw=1)

plt.tight_layout()
plt.show()
print(topic_stats.to_string())

## 7. Model Calibration & Error Analysis

How well does the Bayesian estimate track true mastery?

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

mae  = mean_absolute_error(final_per_student['true_mastery'], final_per_student['final_mastery'])
rmse = mean_squared_error( final_per_student['true_mastery'], final_per_student['final_mastery'], squared=False)
corr = final_per_student[['true_mastery','final_mastery']].corr().iloc[0,1]

print(f'MAE  = {mae:.4f}  (lower is better)')
print(f'RMSE = {rmse:.4f}')
print(f'r    = {corr:.4f}  (Pearson correlation)')

# Calibration curve: bin estimates, compare to actual mastery
bins = np.linspace(0.3, 1.0, 8)
final_per_student['est_bin'] = pd.cut(final_per_student['final_mastery'], bins, include_lowest=True)
cal_df = final_per_student.groupby('est_bin').agg(
    mean_est=('final_mastery', 'mean'),
    mean_true=('true_mastery', 'mean'),
    n=('student_id', 'count'),
).dropna()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot([0.3,1],[0.3,1], 'w--', lw=1.2, alpha=0.4, label='Perfect calibration')
sc = ax1.scatter(cal_df['mean_est'], cal_df['mean_true'],
                 s=cal_df['n']*0.05, c=cal_df['mean_est'],
                 cmap='RdYlGn', vmin=0.3, vmax=1, zorder=5)
ax1.set_xlabel('Mean Estimated Mastery')
ax1.set_ylabel('Mean True Mastery')
ax1.set_title(f'Calibration Curve\nMAE={mae:.3f}, r={corr:.3f}')
plt.colorbar(sc, ax=ax1, label='Mastery estimate')
ax1.legend(fontsize=9)

# Residuals distribution
residuals = final_per_student['final_mastery'] - final_per_student['true_mastery']
ax2.hist(residuals, bins=60, color=ACCENT, alpha=0.7, density=True)
mu, sigma = residuals.mean(), residuals.std()
x = np.linspace(residuals.min(), residuals.max(), 200)
ax2.plot(x, stats.norm.pdf(x, mu, sigma), color=PURPLE, lw=2, label=f'N({mu:.3f},{sigma:.3f})')
ax2.axvline(0, color=GREEN, ls='--', lw=1.5, label='Unbiased')
ax2.set_xlabel('Residual (Estimate − True)')
ax2.set_ylabel('Density')
ax2.set_title('Estimation Residuals')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 8. Mastery Convergence Rate

How quickly does P(θ) stabilise? This justifies adaptive quiz stopping.

In [ ]:
# For each student/topic, compute the running variance of mastery_est
conv_records = []
for (sid, topic), grp in df.groupby(['student_id', 'topic']):
    grp = grp.sort_values('attempt')
    series = grp['mastery_est'].values
    for i in range(1, len(series)):
        window = series[max(0,i-2):i+1]
        variance = np.var(window) if len(window) > 1 else 1.0
        conv_records.append({'student_id': sid, 'topic': topic, 'attempt': i, 'variance': variance})

conv_df = pd.DataFrame(conv_records)
mean_variance = conv_df.groupby('attempt')['variance'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(mean_variance.index, mean_variance.values, color=ACCENT, lw=2.5)
ax.fill_between(mean_variance.index, mean_variance.values, alpha=0.15, color=ACCENT)
ax.axhline(1e-3, color=GREEN, ls='--', lw=1.2, label='Convergence threshold 1e-3')
ax.set_xlabel('Quiz Attempt Number')
ax.set_ylabel('Mean Rolling Variance of Mastery Estimate')
ax.set_title('Mastery Estimate Convergence\n(lower = model is confident)')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

# Find the elbow (where variance drops below threshold)
threshold = 1e-3
elbow = (mean_variance < threshold).idxmax()
print(f'Mastery estimates stabilise around attempt {elbow} (variance < {threshold})')
print(f'\n→ Recommendation: Offer ≥{elbow} quiz questions per topic for reliable mastery scoring.')

## 9. Red → Green Colour Mapping

The node colour in the 3D graph uses `H = mastery × 0.33` (0 = red, 0.33 = green).

In [ ]:
import colorsys

mastery_values = np.linspace(0, 1, 200)
colours = [colorsys.hls_to_rgb(m * 0.33, 0.55, 0.82) for m in mastery_values]

fig, ax = plt.subplots(figsize=(12, 1.8))
for i, (m, c) in enumerate(zip(mastery_values, colours)):
    ax.barh(0, 1/len(mastery_values), left=i/len(mastery_values), color=c, height=1, edgecolor='none')

ax.set_xlim(0, 1); ax.set_ylim(-0.5, 0.5); ax.set_yticks([])
ax.set_xticks([0, 0.5, 0.8, 1.0])
ax.set_xticklabels(['0% (Red\nNo mastery)', '50% (Yellow\nProficient)', '80% (Green\nExpert)', '100%'], fontsize=9)
ax.set_title('Node Colour Mapping: Mastery → HSL(θ×0.33, 0.82, 0.55)', fontsize=11)

# Milestone markers
for m_pct, label in [(0.5, 'Proficient'), (0.8, 'Expert')]:
    ax.axvline(m_pct, color='white', lw=2, ls='--', alpha=0.6)
    ax.text(m_pct, 0.4, label, color='white', fontsize=8, ha='center', va='bottom')

plt.tight_layout()
plt.show()
print('\nApp colour thresholds:')
print(f'  0%  – 49%  : Red/Orange  (Learning)   HSL={0*0.33:.2f}–{0.49*0.33:.2f}')
print(f'  50% – 79%  : Yellow/Lime (Proficient)  HSL={0.5*0.33:.2f}–{0.79*0.33:.2f}')
print(f'  80% – 100% : Green       (Expert)      HSL={0.8*0.33:.2f}–{1.0*0.33:.2f}')

## 10. Save Synthetic Dataset

Export the full dataset for use with the backend or further analysis.

In [ ]:
# Save to CSV
out_path = 'synthetic_mastery_dataset.csv'
df.to_csv(out_path, index=False)
print(f'✅ Saved {len(df):,} rows to {out_path}')

# Summary statistics
print('\n── Dataset summary ────────────────────────────')
print(f'Students           : {df["student_id"].nunique():,}')
print(f'Topics             : {df["topic"].nunique()}')
print(f'Total events       : {len(df):,}')
print(f'Success rate       : {df["success"].mean():.1%}')
print(f'Mean difficulty    : {df["difficulty"].mean():.3f}')
print(f'Mean final mastery : {final_per_student["final_mastery"].mean():.3f}')
print(f'Expert rate (≥0.8) : {(final_per_student["final_mastery"] >= 0.8).mean():.1%}')
print()
print('The Bayesian model is production-ready:')
print(f'  • Calibration r = {corr:.3f}  (excellent at > 0.85)')
print(f'  • MAE           = {mae:.3f}   (typical error is only {mae:.0%} mastery)')
print(f'  • Convergence   ≈ {elbow} quiz attempts per topic')